# Modul 16: LeNet und Transfer Learning mit Keras

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** LeNet mit Keras, Transfer mit Keras  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene Keras-Anwendung für Bilddaten  
    **Orientierungszeit:** etwa 150 bis 210 Minuten

    ## Überblick

    Sie bereiten kleine Bildtensoren vor, bauen ein LeNet-ähnliches CNN und untersuchen Feature-Map-Formen, Training und Fehlerbilder. Anschließend führen Sie ein ressourcenschonendes Transfer-Learning-Experiment mit MobileNetV2 durch.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_16A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_16B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Bilddaten in passende Tensorformen bringen und Pixelwerte normalisieren.
- Ausgabeformen von Faltung, Padding, Stride und Pooling bestimmen.
- Eine LeNet-ähnliche CNN-Architektur in Keras implementieren.
- CNNs ressourcenschonend trainieren und Lernkurven sowie Fehlerbilder analysieren.
- Dropout, Datenaugmentation und Modellgröße als Regularisierungsentscheidungen beurteilen.
- MobileNetV2 laden, eine Basis einfrieren und einen neuen Klassifikationskopf trainieren.
- Transfer- und Scratch-Ansätze mit identischen Daten und nachvollziehbaren Ressourcenkennzahlen vergleichen.

    ## Bewertete Fähigkeiten

    - Bildtensoren, Normalisierung und Feature-Map-Formen
- Conv2D, MaxPooling2D, Flatten und Dense in Keras
- CNN-Training, Lernkurven, Konfusionsmatrix und Fehlerbilder
- Bildaugmentation und Dropout
- MobileNetV2, preprocess_input, Freezing und Transfer-Kopf

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# TensorFlow ist in Google Colab üblicherweise bereits verfügbar.
# Der Fallback installiert nur dann eine CPU-Version, wenn der Import fehlt.
import os
import sys
import subprocess
import warnings
from pathlib import Path

try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu"])
    import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

import time

from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

digits_16 = load_digits()
images_16 = digits_16.images.astype("float32")
labels_16 = digits_16.target.astype("int64")

# Die Originalpixel liegen zwischen 0 und 16. Division durch 16 bringt
# sie in den üblichen Bereich zwischen null und eins.
images_16 = images_16 / 16.0
images_16 = images_16[..., np.newaxis]

X_train_valid_16, X_test_16, y_train_valid_16, y_test_16 = train_test_split(
    images_16,
    labels_16,
    test_size=0.20,
    stratify=labels_16,
    random_state=RANDOM_SEED,
)
X_train_16, X_valid_16, y_train_16, y_valid_16 = train_test_split(
    X_train_valid_16,
    y_train_valid_16,
    test_size=0.25,
    stratify=y_train_valid_16,
    random_state=RANDOM_SEED,
)

# Für das Transfer-Experiment verwenden wir drei Klassen. Das hält den
# Datensatz klein und die Laufzeit auf Colab Free überschaubar.
transfer_mask_16 = np.isin(labels_16, [0, 1, 2])
transfer_images_16 = images_16[transfer_mask_16]
transfer_labels_16 = labels_16[transfer_mask_16]
TX_train_valid_16, TX_test_16, Ty_train_valid_16, Ty_test_16 = train_test_split(
    transfer_images_16,
    transfer_labels_16,
    test_size=0.20,
    stratify=transfer_labels_16,
    random_state=RANDOM_SEED,
)
TX_train_16, TX_valid_16, Ty_train_16, Ty_valid_16 = train_test_split(
    TX_train_valid_16,
    Ty_train_valid_16,
    test_size=0.25,
    stratify=Ty_train_valid_16,
    random_state=RANDOM_SEED,
)

print("LeNet Train/Valid/Test:", X_train_16.shape, X_valid_16.shape, X_test_16.shape)
print("Transfer Train/Valid/Test:", TX_train_16.shape, TX_valid_16.shape, TX_test_16.shape)

print("TensorFlow-Version:", tf.__version__)
print("Schneller Validierungsmodus:", FAST_MODE)


## Aufgabe 1: Bildtensoren und CNN-Ausgabeformen vorbereiten

    Untersuchen und transformieren Sie die Digits-Bilder.

1. Bestätigen Sie Batch-, Höhen-, Breiten- und Kanalachse der Trainingsdaten.
2. Prüfen Sie Wertebereich und Datentyp und visualisieren Sie je ein Beispiel der Klassen 0, 1 und 2.
3. Implementieren Sie eine Funktion für die räumliche Ausgabegröße einer Faltung.
4. Berechnen Sie die Formen nach einer `3 x 3`-Faltung mit `valid`, mit `same` und nach `2 x 2` Max-Pooling.
5. Erzeugen Sie eine einzelne, nicht trainierte `Conv2D`-Schicht mit vier Filtern, wenden Sie sie auf fünf Bilder an und visualisieren Sie die vier Feature Maps des ersten Bildes.

> **Hinweis:** Notieren Sie Formen immer im Schema `(Batch, Höhe, Breite, Kanäle)`.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Notieren Sie Formen immer im Schema `(Batch, Höhe, Breite, Kanäle)`.

## Aufgabe 2: Eine LeNet-ähnliche Architektur in Keras bauen

    Implementieren Sie ein kompaktes LeNet-ähnliches CNN für zehn Ziffernklassen.

1. Verwenden Sie eine Eingabeform `(8, 8, 1)`.
2. Bauen Sie zwei Blöcke aus `Conv2D` mit ReLU und `MaxPooling2D`.
3. Verwenden Sie anschließend `Flatten`, eine kleine Dense-Schicht und eine zehnklassige Softmax-Ausgabe.
4. Kompilieren Sie mit `sparse_categorical_crossentropy`, Adam und Accuracy.
5. Geben Sie die Modellzusammenfassung aus, zählen Sie Parameter und prüfen Sie die Ausgabeform eines Batches.

> **Hinweis:** Verfolgen Sie die Tensorform nach jeder Schicht, bevor Sie Flatten einsetzen.

In [ ]:
# Speichern Sie das Modell als lenet_16 für die nächste Aufgabe.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Verfolgen Sie die Tensorform nach jeder Schicht, bevor Sie Flatten einsetzen.

## Aufgabe 3: LeNet trainieren und Fehlerbilder analysieren

    Trainieren Sie `lenet_16` ressourcenschonend.

1. Verwenden Sie Early Stopping mit Wiederherstellung der besten Gewichte.
2. Trainieren Sie höchstens 30 Epochen mit einem kleinen Batch.
3. Visualisieren Sie Trainings- und Validierungsverlust sowie Accuracy.
4. Bewerten Sie das Modell auf dem Testset und erstellen Sie eine Konfusionsmatrix.
5. Finden Sie bis zu sechs falsch klassifizierte Testbilder und zeigen Sie wahres Label, Vorhersage und maximale Wahrscheinlichkeit.
6. Nennen Sie mindestens zwei plausible Ursachen für typische Verwechslungen.

> **Hinweis:** Nutzen Sie `np.flatnonzero(predictions != labels)`, um Fehlerindizes gezielt zu finden.

In [ ]:
epochs_16 = 5 if FAST_MODE else 30

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Nutzen Sie `np.flatnonzero(predictions != labels)`, um Fehlerindizes gezielt zu finden.

## Aufgabe 4: Regularisierung mit Augmentation und Dropout vergleichen

    Erstellen Sie eine regulierte Alternative zum ersten LeNet-Modell.

1. Verwenden Sie eine kleine Keras-Augmentationspipeline mit Rotation und Translation, die nur im Training aktiv ist.
2. Ergänzen Sie Dropout im dichten Teil des Netzes.
3. Halten Sie die übrige Architektur und die Datenpartitionen möglichst vergleichbar.
4. Trainieren Sie mit Early Stopping.
5. Vergleichen Sie beide Modelle anhand von Parameterzahl, bester Validierungsgenauigkeit und Testgenauigkeit.
6. Entscheiden Sie vorsichtig, ob die Regularisierung in diesem Lauf geholfen hat.

> **Hinweis:** Augmentation gehört in die Modell- oder Trainingspipeline, aber nicht in die Testdaten.

In [ ]:
regularized_epochs_16 = 5 if FAST_MODE else 30

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Augmentation gehört in die Modell- oder Trainingspipeline, aber nicht in die Testdaten.

## Aufgabe 5: Integration: MobileNetV2 einfrieren und fair mit einem kleinen CNN vergleichen

    Führen Sie ein kleines Transfer-Learning-Experiment für die Ziffern 0, 1 und 2 durch.

1. Schreiben Sie eine Vorverarbeitung, die `8 x 8 x 1`-Bilder auf `64 x 64 x 3` bringt.
2. Erstellen Sie ein MobileNetV2-Basismodell ohne Top. Verwenden Sie ImageNet-Gewichte, falls sie verfügbar sind, und sonst einen klar dokumentierten Fallback ohne vortrainierte Gewichte.
3. Frieren Sie die Basis vollständig ein und ergänzen Sie einen Kopf für drei Klassen.
4. Erstellen Sie zusätzlich ein kleines Scratch-CNN für dieselben drei Klassen.
5. Trainieren Sie beide Modelle mit denselben Splits und derselben maximalen Epochenzahl.
6. Vergleichen Sie beste Validierungsgenauigkeit, Testgenauigkeit, trainierbare Parameter und Inferenzzeit auf demselben Teststapel.

Halten Sie die Basis eingefroren. Fine-Tuning ist noch nicht erforderlich.

> **Hinweis:** Prüfen Sie `base_model.trainable` und zählen Sie nur trainierbare Parameter.

In [ ]:
transfer_epochs_16 = 1 if FAST_MODE else 3
transfer_batch_size_16 = 32

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Prüfen Sie `base_model.trainable` und zählen Sie nur trainierbare Parameter.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.